# LLMTIME Tokenization Prediction Pipeline

This notebook implements the PDF section 3.1.2 LLMTIME idea: convert continuous price values into standardized digit strings so a language-model backend can treat forecasting as next-token continuation.

Concept mapping:

- Fixed precision truncation: close values are scaled and converted to fixed-width integer strings.
- Percentile rescaling: train-period close prices are centered by median and scaled by an alpha-percentile absolute deviation, not max value.
- Digit separation: `style="gpt"` injects spaces between digits; `style="llama"` keeps compact digit strings.
- Backend: `local` runs a deterministic local trend continuation so the notebook works without API keys; `api` calls an OpenAI-compatible chat/completion endpoint from environment variables.


In [ ]:
from pathlib import Path
import importlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import Image, display

SRC_DIR = Path.cwd() / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import llmtime_backend
import plot_results

importlib.reload(llmtime_backend)
importlib.reload(plot_results)

from benchmark_utils import INDEX_TICKERS
from llmtime_backend import LLMTIMEConfig, build_llmtime_prompt, download_index, fit_llmtime_scaler, run_llmtime_benchmark
from plot_results import plot_price_predictions

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)


## Configuration

In [ ]:
START = "2010-01-01"
END = "2026-06-01"
TRAIN_END = "2025-12-31"
TEST_START = "2026-01-01"
TEST_END = "2026-05-31"

LOOKBACK = 30
HORIZON = 1
PRECISION = 2
ALPHA = 0.95
TOKEN_STYLE = "gpt"  # "gpt" inserts spaces between digits; "llama" keeps compact numbers.
BACKEND = "local"    # Use "api" after setting LLMTIME_API_URL, LLMTIME_API_KEY, and LLMTIME_MODEL.

OUTPUT_DIR = Path(f"outputs_llmtime_{BACKEND}_h{HORIZON}")
PLOTS_DIR = Path(f"plots_llmtime_{BACKEND}_h{HORIZON}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

RUN_PREDICTION = True
config = LLMTIMEConfig(precision=PRECISION, alpha=ALPHA, style=TOKEN_STYLE)


## Inspect One LLMTIME Prompt

In [ ]:
example_index = "Nasdaq"
raw = download_index(INDEX_TICKERS[example_index], start=START, end=END)
train_values = raw.loc[:TRAIN_END, "Close"].to_numpy(dtype=float)
scaler = fit_llmtime_scaler(train_values, precision=PRECISION, alpha=ALPHA)
history = raw["Close"].dropna().iloc[-LOOKBACK:].to_numpy(dtype=float)
prompt = build_llmtime_prompt(history, scaler=scaler, horizon=HORIZON, config=config)

print("Scaler offset:", scaler.offset)
print("Scaler scale:", scaler.scale)
print(prompt[:1200])


## Run LLMTIME Backend And Build Benchmark Outputs

In [ ]:
if RUN_PREDICTION:
    results = []
    for index_name, ticker in INDEX_TICKERS.items():
        print(f"Running LLMTIME-{BACKEND} {index_name} ({ticker})...")
        result = run_llmtime_benchmark(
            name=index_name,
            ticker=ticker,
            start=START,
            end=END,
            train_end=TRAIN_END,
            test_start=TEST_START,
            test_end=TEST_END,
            lookback=LOOKBACK,
            horizon=HORIZON,
            output_dir=OUTPUT_DIR,
            backend=BACKEND,
            config=config,
        )
        results.append(result)
        print(
            f"{index_name}: RMSE={result['rmse']:.2f}, MAE={result['mae']:.2f}, "
            f"MAPE={result['mape_pct']:.2f}%, Direction={result['direction_accuracy_pct']:.2f}%"
        )

    summary = pd.DataFrame(results)
    summary.to_csv(OUTPUT_DIR / "benchmark_summary.csv", index=False)
    (OUTPUT_DIR / "benchmark_summary.json").write_text(json.dumps(results, indent=2), encoding="utf-8")

summary = pd.read_csv(OUTPUT_DIR / "benchmark_summary.csv")
summary


## Benchmark Summary

In [ ]:
metric_cols = [
    "index", "ticker", "model", "samples", "train_samples", "rmse", "mae", "mape_pct",
    "naive_rmse", "naive_mae", "rmse_vs_naive_pct", "mae_vs_naive_pct",
    "direction_accuracy_pct", "tokenizer", "precision", "alpha",
]
summary[metric_cols]


## Inspect First Prompt Saved For Each Index

In [ ]:
first_prompt_path = OUTPUT_DIR / "Nasdaq_first_prompt.txt"
print(first_prompt_path.read_text(encoding="utf-8")[:1200])


## Generate Price Prediction Plots

In [ ]:
for index_name in summary["index"]:
    plot_price_predictions(
        index_name=index_name,
        prediction_path=OUTPUT_DIR / f"{index_name}_predictions.csv",
        output_path=PLOTS_DIR / f"{index_name}_price_prediction.png",
    )

print(f"Saved plots to {PLOTS_DIR.resolve()}")


## Display Example Plot

In [ ]:
example_plot = PLOTS_DIR / "Nasdaq_price_prediction.png"
if example_plot.exists():
    display(Image(filename=str(example_plot)))
else:
    print(f"Plot not found: {example_plot}")
